In [ ]:
# In[1]:

import pandas as pd

# Load files
metric = pd.read_csv("metric.csv")
trace = pd.read_csv("trace.csv")
log = pd.read_csv("log.csv")

# Parse timestamps as UTC
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)
trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)
log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)

# 1) Unique cmdb_id values and counts (per file)
metric_cmdb_counts = metric['cmdb_id'].value_counts().rename_axis('cmdb_id').reset_index(name='rows').sort_values('rows', ascending=False).reset_index(drop=True)
trace_cmdb_counts  = trace['cmdb_id'].value_counts().rename_axis('cmdb_id').reset_index(name='rows').sort_values('rows', ascending=False).reset_index(drop=True)
log_cmdb_counts    = log['cmdb_id'].value_counts().rename_axis('cmdb_id').reset_index(name='rows').sort_values('rows', ascending=False).reset_index(drop=True)

# 2) Unique KPI/trace/log names and counts (per file)
metric_name_counts = metric['kpi_name'].value_counts().rename_axis('kpi_name').reset_index(name='rows').sort_values('rows', ascending=False).reset_index(drop=True)
trace_name_counts  = trace['trace_name'].value_counts().rename_axis('trace_name').reset_index(name='rows').sort_values('rows', ascending=False).reset_index(drop=True)
log_name_counts    = log['log_name'].value_counts().rename_axis('log_name').reset_index(name='rows').sort_values('rows', ascending=False).reset_index(drop=True)

# Aggregations
# metric: per (cmdb_id, kpi_name) compute global P50, P90, P95 and total count
metric_aggregates = (
    metric.groupby(['cmdb_id', 'kpi_name'], observed=True)['value']
    .agg(
        count='count',
        p50=lambda x: x.quantile(0.5),
        p90=lambda x: x.quantile(0.9),
        p95=lambda x: x.quantile(0.95)
    )
    .reset_index()
    .sort_values('count', ascending=False)
    .head(50)
    .reset_index(drop=True)
)

# trace: per (cmdb_id, trace_name) compute global P95 and count
trace_aggregates = (
    trace.groupby(['cmdb_id', 'trace_name'], observed=True)['value']
    .agg(
        count='count',
        p95=lambda x: x.quantile(0.95)
    )
    .reset_index()
    .sort_values('count', ascending=False)
    .head(50)
    .reset_index(drop=True)
)

# log: per (cmdb_id, log_name) compute global P95 and count
log_aggregates = (
    log.groupby(['cmdb_id', 'log_name'], observed=True)['value']
    .agg(
        count='count',
        p95=lambda x: x.quantile(0.95)
    )
    .reset_index()
    .sort_values('count', ascending=False)
    .head(50)
    .reset_index(drop=True)
)

# Keep displayed outputs compact: show top rows for counts and the aggregate tables
metric_cmdb_counts.head(50), metric_name_counts.head(50), metric_aggregates, trace_cmdb_counts.head(50), trace_name_counts.head(50), trace_aggregates, log_cmdb_counts.head(50), log_name_counts.head(50), log_aggregates

```
Out[1]:
```
```python
summary = (
    "Summary of telemetry aggregates (compact):\n\n"
    "Metrics (metric.csv):\n"
    "- Unique cmdb_id top counts: adservice 175, currencyservice 175, recommendationservice 175, emailservice 175, "
    "checkoutservice 150, cartservice 150, paymentservice 150, frontend 150, shippingservice 150, productcatalogservice 150, redis 100, frontend-external 50.\n"
    "- KPI name counts: cpu/workload/socket/mem each 275 rows; latency-50 and latency-90 250 rows; diskio 100; error 50.\n"
    "- Example metric aggregates (per cmdb_id, kpi_name): adservice diskio P95 ≈ 93,255; adservice mem P95 ≈ 1.0622e+08; "
    "many (cmdb_id,kpi) groups have count 25 (shown in the top-50 table).\n\n"
    "Traces (trace.csv):\n"
    "- Unique cmdb_id top counts: checkoutservice 800, frontendservice 700, root 700, recommendationservice 500, productcatalogservice 400, currencyservice 300, emailservice 200, paymentservice 200.\n"
    "- Trace name coverage: many trace.* names (e.g., trace.from_root.duration_mean/p95/error_rate/row_count) with varying counts; 44 distinct trace names shown.\n"
    "- Example trace aggregates: checkoutservice trace.from_checkoutservice.row_count P95 = 298.6; checkoutservice trace.from_frontendservice.duration_mean P95 ≈ 0.9211.\n\n"
    "Logs (log.csv):\n"
    "- Unique cmdb_id top counts: adservice/cartservice/checkoutservice/currencyservice/emailservice/frontend/paymentservice/recommendationservice/shippingservice each 50, redis 10.\n"
    "- Log names: log.error_count and log.row_count have 230 rows each.\n"
    "- Example log aggregates: frontend log.row_count P95 ≈ 2766.8, currencyservice log.row_count P95 ≈ 2446.8, cartservice log.row_count P95 ≈ 805.8; log.error_count P95 = 0.0 for shown services.\n\n"
    "Overall: the tables computed the requested per-(cmdb_id, name) statistics (metric: P50/P90/P95/count; trace/log: P95/count). "
    "Top-50 rows per table were produced; notable high P95 values include adservice diskio and very large memory values, and very large log.row_count P95 for frontend and currencyservice."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id  rows
0               adservice   175
1         currencyservice   175
2   recommendationservice   175
3            emailservice   175
4         checkoutservice   150
5             cartservice   150
6          paymentservice   150
7                frontend   150
8         shippingservice   150
9   productcatalogservice   150
10                  redis   100
11      frontend-external    50,      kpi_name  rows
0         cpu   275
1    workload   275
2      socket   275
3         mem   275
4  latency-50   250
5  latency-90   250
6      diskio   100
7       error    50,                   cmdb_id    kpi_name  count           p50           p90           p95
0               adservice         cpu     25  7.135240e-01  7.855389e-01  7.966259e-01
1               adservice      diskio     25  6.122027e+04  7.575202e+04  9.325574e+04
2               adservice  latency-50     25  2.999723e-03  3.008124e-03  3.008702e-03
3               adservice  latency-90     25  4.609118e-03  4.623893e-03  4.629082e-03
4               adservice         mem     25  1.060223e+08  1.061997e+08  1.062224e+08
..                    ...         ...    ...           ...           ...           ...
45         paymentservice      socket     25  3.000000e+00  3.000000e+00  3.000000e+00
46         paymentservice    workload     25  9.389000e-01  1.186540e+00  1.222993e+00
47  productcatalogservice         cpu     25  3.449035e+00  3.667728e+00  3.754278e+00
48  productcatalogservice  latency-50     25  1.913683e-03  1.999393e-03  2.047555e-03
49  productcatalogservice  latency-90     25  4.398893e-03  4.431730e-03  4.440021e-03

[50 rows x 6 columns],                  cmdb_id  rows
0        checkoutservice   800
1        frontendservice   700
2                   root   700
3  recommendationservice   500
4  productcatalogservice   400
5        currencyservice   300
6           emailservice   200
7         paymentservice   200,                                         trace_name  rows
0                    trace.from_root.duration_mean   175
1                     trace.from_root.duration_p95   175
2                       trace.from_root.error_rate   175
3                        trace.from_root.row_count   175
4             trace.from_checkoutservice.row_count   125
..                                             ...   ...
39          trace.to_frontendservice.duration_mean    50
40  trace.from_recommendationservice.duration_mean    50
41   trace.from_recommendationservice.duration_p95    50
42     trace.from_recommendationservice.error_rate    50
43      trace.from_recommendationservice.row_count    50

[44 rows x 2 columns],             cmdb_id                                trace_name  count         p95
0   checkoutservice  trace.from_checkoutservice.duration_mean     25    0.076209
1   checkoutservice   trace.from_checkoutservice.duration_p95     25    0.117588
2   checkoutservice     trace.from_checkoutservice.error_rate     25    0.000000
3   checkoutservice      trace.from_checkoutservice.row_count     25  298.600000
4   checkoutservice  trace.from_frontendservice.duration_mean     25    0.921095
..              ...                                       ...    ...         ...
45     emailservice   trace.from_checkoutservice.duration_p95     25    0.001011
46     emailservice     trace.from_checkoutservice.error_rate     25    0.000000
47     emailservice      trace.from_checkoutservice.row_count     25   36.800000
48     emailservice             trace.from_root.duration_mean     25    0.000086
49     emailservice              trace.from_root.duration_p95     25    0.000130

[50 rows x 4 columns],                  cmdb_id  rows
0              adservice    50
1            cartservice    50
2        checkoutservice    50
3        currencyservice    50
4           emailservice    50
5               frontend    50
6         paymentservice    50
7  recommendationservice    50
8        shippingservice    50
9                  redis    10,           log_name  rows
0  log.error_count   230
1    log.row_count   230,                   cmdb_id         log_name  count     p95
0               adservice  log.error_count     25     0.0
1               adservice    log.row_count     25   463.8
2             cartservice  log.error_count     25     0.0
3             cartservice    log.row_count     25   805.8
4         checkoutservice  log.error_count     25     0.0
5         checkoutservice    log.row_count     25   110.4
6         currencyservice  log.error_count     25     0.0
7         currencyservice    log.row_count     25  2446.8
8            emailservice  log.error_count     25     0.0
9            emailservice    log.row_count     25    36.8
10               frontend  log.error_count     25     0.0
11               frontend    log.row_count     25  2766.8
12         paymentservice  log.error_count     25     0.0
13         paymentservice    log.row_count     25    73.6
14  recommendationservice  log.error_count     25     0.0
15  recommendationservice    log.row_count     25   585.0
16        shippingservice    log.row_count     25   530.8
17        shippingservice  log.error_count     25     0.0
18                  redis    log.row_count      5     5.0
19                  redis  log.error_count      5     0.0)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd

# Reuse existing metric DataFrame if present; otherwise load
try:
    metric
except NameError:
    metric = pd.read_csv("metric.csv")

# Parse timestamps as UTC (rule 10)
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

# 1) Compute global P95 per (cmdb_id, kpi_name) using the full series (must be done BEFORE filtering) (rule 9)
global_p95_df = (
    metric
    .groupby(['cmdb_id', 'kpi_name'], observed=True)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'global_p95'})
)

# Incident window (UTC)
start = pd.to_datetime('2024-01-17T03:12:50Z', utc=True)
end   = pd.to_datetime('2024-01-17T03:42:50Z', utc=True)

# 2) Filter metric rows to the incident window
metric_window = metric[(metric['timestamp'] >= start) & (metric['timestamp'] <= end)].copy()

# total number of metric rows in the incident window
total_rows = metric_window.shape[0]

# 3) Attach global_p95 to each row in the window
metric_window = metric_window.merge(global_p95_df, on=['cmdb_id', 'kpi_name'], how='left')

# 4) Find anomaly rows where value > global_p95 (strict greater-than)
anoms = metric_window[metric_window['value'] > metric_window['global_p95']].copy()

# 5) Aggregate anomalies per (cmdb_id, kpi_name)
if not anoms.empty:
    agg = (
        anoms
        .groupby(['cmdb_id', 'kpi_name'], observed=True)
        .agg(
            anomaly_count = ('value', 'count'),
            earliest_anom_ts = ('timestamp', 'min'),
            max_value_in_window = ('value', 'max'),
            global_p95 = ('global_p95', 'first')
        )
        .reset_index()
    )
    # Format earliest_anom_ts as ISO UTC strings with 'Z'
    agg['earliest_anomaly_timestamp'] = agg['earliest_anom_ts'].dt.tz_convert('UTC').dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    agg = agg.drop(columns=['earliest_anom_ts'])
    # Sort by anomaly_count desc then by max_value_in_window desc, limit to top 20
    anomaly_summary = agg.sort_values(['anomaly_count', 'max_value_in_window'], ascending=[False, False]).head(20).reset_index(drop=True)
    # Ensure column order
    anomaly_summary = anomaly_summary[['cmdb_id', 'kpi_name', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window', 'global_p95']]
else:
    # Empty result with correct columns
    anomaly_summary = pd.DataFrame(columns=['cmdb_id', 'kpi_name', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window', 'global_p95'])

# Display compact results: anomaly_summary (top 20) and total_rows
anomaly_summary, total_rows

```
Out[2]:
```
Summary of metric anomalies in the incident window (2024-01-17T03:12:50Z — 2024-01-17T03:42:50Z):

- Total metric rows in the incident window: 1750.

- Top anomaly groups (breaching global P95, sorted by anomaly count and max value; limited to top 20):
  1. checkoutservice — mem: 2 breaches; earliest 2024-01-17T03:30:00Z; max 2.684355e+08; global P95 2.646119e+08
  2. adservice — mem: 2 breaches; earliest 2024-01-17T03:37:00Z; max 1.064308e+08; global P95 1.062224e+08
  3. currencyservice — mem: 2 breaches; earliest 2024-01-17T03:34:00Z; max 4.539208e+07; global P95 4.526718e+07
  4. recommendationservice — mem: 2 breaches; earliest 2024-01-17T03:18:00Z; max 4.534600e+07; global P95 4.532437e+07
  5. cartservice — mem: 2 breaches; earliest 2024-01-17T03:31:00Z; max 4.358970e+07; global P95 4.358347e+07
  6. emailservice — mem: 2 breaches; earliest 2024-01-17T03:38:00Z; max 4.328018e+07; global P95 4.327337e+07
  7. paymentservice — mem: 2 breaches; earliest 2024-01-17T03:33:00Z; max 3.899017e+07; global P95 3.859993e+07
  8. frontend — mem: 2 breaches; earliest 2024-01-17T03:28:00Z; max 1.612520e+07; global P95 1.606241e+07
  9. productcatalogservice — mem: 2 breaches; earliest 2024-01-17T03:19:00Z; max 1.062632e+07; global P95 1.056094e+07
 10. shippingservice — mem: 2 breaches; earliest 2024-01-17T03:38:00Z; max 8.781756e+06; global P95 8.690499e+06
 11. redis — mem: 2 breaches; earliest 2024-01-17T03:37:00Z; max 3.043328e+06; global P95 3.039068e+06
 12. adservice — diskio: 2 breaches; earliest 2024-01-17T03:15:00Z; max 9.811943e+04; global P95 9.325574e+04
 13. redis — diskio: 2 breaches; earliest 2024-01-17T03:18:00Z; max 1.991381e+04; global P95 1.451600e+04
 14. productcatalogservice — workload: 2 breaches; earliest 2024-01-17T03:15:00Z; max 159.7802; global P95 149.4132
 15. currencyservice — workload: 2 breaches; earliest 2024-01-17T03:15:00Z; max 84.0136; global P95 83.16429
 16. currencyservice — cpu: 2 breaches; earliest 2024-01-17T03:15:00Z; max 31.56382; global P95 30.41169
 17. cartservice — workload: 2 breaches; earliest 2024-01-17T03:15:00Z; max 29.06; global P95 26.53869
 18. recommendationservice — workload: 2 breaches; earliest 2024-01-17T03:15:00Z; max 20.7998; global P95 19.53193
 19. checkoutservice — cpu: 2 breaches; earliest 2024-01-17T03:33:00Z; max 19.86398; global P95 19.72638
 20. adservice — workload: 2 breaches; earliest 2024-01-17T03:15:00Z; max 16.6197; global P95 15.37333

- Interpretation: Memory (mem) breaches are the most widespread in this window, affecting many services (checkoutservice, adservice, currencyservice, recommendationservice, cartservice, emailservice, paymentservice, frontend, productcatalogservice, shippingservice, redis). There are also notable disk I/O breaches (adservice, redis) and workload/cpu breaches for several services. Each reported group had 2 rows in the window exceeding its global P95.

If you want, I can provide the full anomaly table exported as a DataFrame view (head/rows) or additional context for any specific cmdb_id/kpi_name.

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id  kpi_name  anomaly_count earliest_anomaly_timestamp  max_value_in_window    global_p95
0         checkoutservice       mem              2       2024-01-17T03:30:00Z         2.684355e+08  2.646119e+08
1               adservice       mem              2       2024-01-17T03:37:00Z         1.064308e+08  1.062224e+08
2         currencyservice       mem              2       2024-01-17T03:34:00Z         4.539208e+07  4.526718e+07
3   recommendationservice       mem              2       2024-01-17T03:18:00Z         4.534600e+07  4.532437e+07
4             cartservice       mem              2       2024-01-17T03:31:00Z         4.358970e+07  4.358347e+07
5            emailservice       mem              2       2024-01-17T03:38:00Z         4.328018e+07  4.327337e+07
6          paymentservice       mem              2       2024-01-17T03:33:00Z         3.899017e+07  3.859993e+07
7                frontend       mem              2       2024-01-17T03:28:00Z         1.612520e+07  1.606241e+07
8   productcatalogservice       mem              2       2024-01-17T03:19:00Z         1.062632e+07  1.056094e+07
9         shippingservice       mem              2       2024-01-17T03:38:00Z         8.781756e+06  8.690499e+06
10                  redis       mem              2       2024-01-17T03:37:00Z         3.043328e+06  3.039068e+06
11              adservice    diskio              2       2024-01-17T03:15:00Z         9.811943e+04  9.325574e+04
12                  redis    diskio              2       2024-01-17T03:18:00Z         1.991381e+04  1.451600e+04
13  productcatalogservice  workload              2       2024-01-17T03:15:00Z         1.597802e+02  1.494132e+02
14        currencyservice  workload              2       2024-01-17T03:15:00Z         8.401360e+01  8.316429e+01
15        currencyservice       cpu              2       2024-01-17T03:15:00Z         3.156382e+01  3.041169e+01
16            cartservice  workload              2       2024-01-17T03:15:00Z         2.906000e+01  2.653869e+01
17  recommendationservice  workload              2       2024-01-17T03:15:00Z         2.079980e+01  1.953193e+01
18        checkoutservice       cpu              2       2024-01-17T03:33:00Z         1.986398e+01  1.972638e+01
19              adservice  workload              2       2024-01-17T03:15:00Z         1.661970e+01  1.537333e+01, 1750)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse existing trace DataFrame if present; otherwise load
try:
    trace
except NameError:
    trace = pd.read_csv("trace.csv")

# Parse timestamps as UTC
trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)

# Compute global P95 and P5 per (cmdb_id, trace_name) using full series (do BEFORE filtering)
grp = trace.groupby(['cmdb_id', 'trace_name'], observed=True)['value']
global_q = grp.quantile([0.95, 0.05]).unstack(level=-1)
# quantile produced columns named 0.95 and 0.05; rename them
global_q = global_q.rename(columns={0.95: 'global_p95', 0.05: 'global_p5'}).reset_index()

# Incident window (UTC)
start = pd.to_datetime('2024-01-17T03:12:50Z', utc=True)
end   = pd.to_datetime('2024-01-17T03:42:50Z', utc=True)

# Filter trace rows to the incident window
trace_window = trace[(trace['timestamp'] >= start) & (trace['timestamp'] <= end)].copy()

# Total number of trace rows in the incident window
total_rows = trace_window.shape[0]

# Attach global thresholds to window rows
trace_window = trace_window.merge(global_q, on=['cmdb_id', 'trace_name'], how='left')

# Identify high anomalies: trace_name contains 'duration' or 'error_rate' and value > global_p95
mask_high = trace_window['trace_name'].str.contains('duration|error_rate', case=False, regex=True) & (trace_window['value'] > trace_window['global_p95'])

# Identify low anomalies: trace_name contains 'row_count' and value < global_p5
mask_low = trace_window['trace_name'].str.contains('row_count', case=False, regex=True) & (trace_window['value'] < trace_window['global_p5'])

high_anoms = trace_window[mask_high].copy()
low_anoms = trace_window[mask_low].copy()

# Aggregate high anomalies per (cmdb_id, trace_name)
if not high_anoms.empty:
    agg_high = (
        high_anoms
        .groupby(['cmdb_id', 'trace_name'], observed=True)
        .agg(
            anomaly_count = ('value', 'count'),
            earliest_ts = ('timestamp', 'min'),
            extreme_value_in_window = ('value', 'max'),
            global_p95 = ('global_p95', 'first'),
            global_p5 = ('global_p5', 'first')
        )
        .reset_index()
    )
    agg_high['direction'] = 'high'
    # severity = extreme / global_p95 (higher means more severe)
    agg_high['severity'] = agg_high['extreme_value_in_window'] / agg_high['global_p95']
else:
    agg_high = pd.DataFrame(columns=['cmdb_id','trace_name','anomaly_count','earliest_ts','extreme_value_in_window','global_p95','global_p5','direction','severity'])

# Aggregate low anomalies per (cmdb_id, trace_name)
if not low_anoms.empty:
    agg_low = (
        low_anoms
        .groupby(['cmdb_id', 'trace_name'], observed=True)
        .agg(
            anomaly_count = ('value', 'count'),
            earliest_ts = ('timestamp', 'min'),
            extreme_value_in_window = ('value', 'min'),
            global_p95 = ('global_p95', 'first'),
            global_p5 = ('global_p5', 'first')
        )
        .reset_index()
    )
    agg_low['direction'] = 'low'
    # severity = global_p5 / extreme (higher means more severe drop); guard divide-by-zero
    agg_low['severity'] = agg_low.apply(lambda r: (r['global_p5'] / r['extreme_value_in_window']) if r['extreme_value_in_window'] != 0 else np.inf, axis=1)
else:
    agg_low = pd.DataFrame(columns=['cmdb_id','trace_name','anomaly_count','earliest_ts','extreme_value_in_window','global_p95','global_p5','direction','severity'])

# Combine, format earliest timestamp, sort by anomaly_count desc then severity desc, limit to top 20
anomaly_df = pd.concat([agg_high, agg_low], ignore_index=True, sort=False)
if not anomaly_df.empty:
    anomaly_df['earliest_anomaly_timestamp'] = pd.to_datetime(anomaly_df['earliest_ts']).dt.tz_convert('UTC').dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    anomaly_df = anomaly_df.drop(columns=['earliest_ts'])
    anomaly_summary = (
        anomaly_df
        .sort_values(['anomaly_count', 'severity'], ascending=[False, False])
        .head(20)
        .reset_index(drop=True)
    )
    # Keep requested column order
    anomaly_summary = anomaly_summary[['cmdb_id', 'trace_name', 'anomaly_count', 'direction', 'earliest_anomaly_timestamp', 'extreme_value_in_window', 'global_p95', 'global_p5']]
else:
    anomaly_summary = pd.DataFrame(columns=['cmdb_id', 'trace_name', 'anomaly_count', 'direction', 'earliest_anomaly_timestamp', 'extreme_value_in_window', 'global_p95', 'global_p5'])

# Display compact results: anomaly_summary (top 20) and total_rows
anomaly_summary, total_rows

```
Out[3]:
```
```python
summary = (
    "Summary of trace anomalies in the incident window (2024-01-17T03:12:50Z — 2024-01-17T03:42:50Z):\n\n"
    "- Total trace rows in the window: 3800.\n\n"
    "- Anomaly type: All detected anomalies are 'low' row_count anomalies (value < global P5). "
    "No 'high' anomalies were reported for duration or error_rate.\n\n"
    "- Timing and scope: Most anomaly groups have 2 breaches each, with the earliest anomaly timestamp at 2024-01-17T03:15:00Z. "
    "This affects many inter-service trace row_count metrics (e.g., from_frontend, to_checkoutservice, from_root, to_productcatalogservice, etc.).\n\n"
    "- Representative examples:\n"
    "  * checkoutservice — trace.from_frontendservice.row_count: extreme=3.0 vs global_p5=18.6\n"
    "  * frontendservice — trace.to_checkoutservice.row_count: extreme=3.0 vs global_p5=18.6\n"
    "  * checkoutservice — trace.from_checkoutservice.row_count: extreme=26.0 vs global_p5=157.8\n"
    "  * frontendservice/productcatalogservice — trace.to_productcatalogservice / trace.from_frontendservice.row_count: extreme=616.0 vs global_p5=3023.4\n"
    "  * emailservice/paymentservice/productcatalogservice/recommendationservice — trace.from_root.row_count: extremes 4/2/4/4 vs global_p5s 20.8/10.4/20.8/20.8\n\n"
    "- Interpretation: The signal indicates a widespread drop in trace row counts (reduced request/trace throughput) across multiple services around 03:15:00Z, rather than increases in latency or error rates. "
    "This points toward reduced traffic or request loss impacting many service-to-service calls during that time window.\n\n"
    "If you want, I can correlate these trace row_count drops with metric (e.g., cpu/mem/diskio) or log data for the same window to narrow root cause."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id                                  trace_name  anomaly_count direction earliest_anomaly_timestamp  extreme_value_in_window  global_p95  global_p5
0         checkoutservice        trace.from_frontendservice.row_count              2       low       2024-01-17T03:15:00Z                      3.0        36.8       18.6
1         frontendservice          trace.to_checkoutservice.row_count              2       low       2024-01-17T03:15:00Z                      3.0        36.8       18.6
2         checkoutservice        trace.from_checkoutservice.row_count              2       low       2024-01-17T03:15:00Z                     26.0       298.6      157.8
3         checkoutservice          trace.to_checkoutservice.row_count              2       low       2024-01-17T03:15:00Z                     26.0       298.6      157.8
4            emailservice                   trace.from_root.row_count              2       low       2024-01-17T03:15:00Z                      4.0        24.0       20.8
5          paymentservice                   trace.from_root.row_count              2       low       2024-01-17T03:15:00Z                      2.0        12.0       10.4
6   productcatalogservice                   trace.from_root.row_count              2       low       2024-01-17T03:15:00Z                      4.0        24.0       20.8
7   recommendationservice                   trace.from_root.row_count              2       low       2024-01-17T03:15:00Z                      4.0        24.0       20.8
8                    root             trace.to_emailservice.row_count              2       low       2024-01-17T03:15:00Z                      4.0        24.0       20.8
9                    root           trace.to_paymentservice.row_count              2       low       2024-01-17T03:15:00Z                      2.0        12.0       10.4
10                   root    trace.to_productcatalogservice.row_count              2       low       2024-01-17T03:15:00Z                      4.0        24.0       20.8
11                   root    trace.to_recommendationservice.row_count              2       low       2024-01-17T03:15:00Z                      4.0        24.0       20.8
12        frontendservice    trace.to_recommendationservice.row_count              2       low       2024-01-17T03:15:00Z                     92.0       585.0      466.6
13  productcatalogservice  trace.from_recommendationservice.row_count              2       low       2024-01-17T03:15:00Z                     92.0       585.0      466.6
14  recommendationservice        trace.from_frontendservice.row_count              2       low       2024-01-17T03:15:00Z                     92.0       585.0      466.6
15  recommendationservice  trace.from_recommendationservice.row_count              2       low       2024-01-17T03:15:00Z                     92.0       585.0      466.6
16  recommendationservice    trace.to_productcatalogservice.row_count              2       low       2024-01-17T03:15:00Z                     92.0       585.0      466.6
17  recommendationservice    trace.to_recommendationservice.row_count              2       low       2024-01-17T03:15:00Z                     92.0       585.0      466.6
18        frontendservice    trace.to_productcatalogservice.row_count              2       low       2024-01-17T03:15:00Z                    616.0      3819.8     3023.4
19  productcatalogservice        trace.from_frontendservice.row_count              2       low       2024-01-17T03:15:00Z                    616.0      3819.8     3023.4, 3800)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse existing log/error_logs if present, otherwise load
try:
    log
except NameError:
    log = pd.read_csv("log.csv")
try:
    error_logs
except NameError:
    error_logs = pd.read_csv("error_logs.csv")

# Parse timestamps as UTC
log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)
error_logs['timestamp'] = pd.to_datetime(error_logs['timestamp'], unit='s', utc=True)

# Compute global P95 and P5 per (cmdb_id, log_name) using full series (must be before filtering)
grp = log.groupby(['cmdb_id', 'log_name'], observed=True)['value']
global_q = grp.quantile([0.95, 0.05]).unstack(level=-1).rename(columns={0.95: 'global_p95', 0.05: 'global_p5'}).reset_index()

# Incident window (UTC)
start = pd.to_datetime('2024-01-17T03:12:50Z', utc=True)
end   = pd.to_datetime('2024-01-17T03:42:50Z', utc=True)

# Filter log rows to the incident window
log_window = log[(log['timestamp'] >= start) & (log['timestamp'] <= end)].copy()

# Attach global thresholds to window rows
log_window = log_window.merge(global_q, on=['cmdb_id', 'log_name'], how='left')

# Identify anomalies:
# (1) log.row_count values < global_P5  => direction 'low'
mask_low = log_window['log_name'].str.contains('row_count', case=False, regex=True) & (log_window['value'] < log_window['global_p5'])

# (2) log.error_count values > global_P95 => direction 'high'
mask_high = log_window['log_name'].str.contains('error_count', case=False, regex=True) & (log_window['value'] > log_window['global_p95'])

low_anoms = log_window[mask_low].copy()
high_anoms = log_window[mask_high].copy()

# Aggregate high anomalies
if not high_anoms.empty:
    agg_high = (
        high_anoms
        .groupby(['cmdb_id', 'log_name'], observed=True)
        .agg(
            anomaly_count = ('value', 'count'),
            earliest_ts = ('timestamp', 'min'),
            extreme_value_in_window = ('value', 'max'),
            global_p95 = ('global_p95', 'first'),
            global_p5 = ('global_p5', 'first')
        )
        .reset_index()
    )
    agg_high['direction'] = 'high'
    agg_high['severity'] = agg_high.apply(lambda r: (r['extreme_value_in_window'] / r['global_p95']) if pd.notna(r['global_p95']) and r['global_p95'] != 0 else np.inf, axis=1)
else:
    agg_high = pd.DataFrame(columns=['cmdb_id','log_name','anomaly_count','earliest_ts','extreme_value_in_window','global_p95','global_p5','direction','severity'])

# Aggregate low anomalies
if not low_anoms.empty:
    agg_low = (
        low_anoms
        .groupby(['cmdb_id', 'log_name'], observed=True)
        .agg(
            anomaly_count = ('value', 'count'),
            earliest_ts = ('timestamp', 'min'),
            extreme_value_in_window = ('value', 'min'),
            global_p95 = ('global_p95', 'first'),
            global_p5 = ('global_p5', 'first')
        )
        .reset_index()
    )
    agg_low['direction'] = 'low'
    agg_low['severity'] = agg_low.apply(lambda r: (r['global_p5'] / r['extreme_value_in_window']) if pd.notna(r['global_p5']) and r['extreme_value_in_window'] != 0 else np.inf, axis=1)
else:
    agg_low = pd.DataFrame(columns=['cmdb_id','log_name','anomaly_count','earliest_ts','extreme_value_in_window','global_p95','global_p5','direction','severity'])

# Combine and format
anomaly_df = pd.concat([agg_high, agg_low], ignore_index=True, sort=False)
if not anomaly_df.empty:
    anomaly_df['earliest_anomaly_timestamp'] = pd.to_datetime(anomaly_df['earliest_ts']).dt.tz_convert('UTC').dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    anomaly_df = anomaly_df.drop(columns=['earliest_ts'])
    anomaly_summary = (
        anomaly_df
        .sort_values(['anomaly_count', 'severity'], ascending=[False, False])
        .head(20)
        .reset_index(drop=True)
    )
    anomaly_summary = anomaly_summary[['cmdb_id', 'log_name', 'anomaly_count', 'direction', 'earliest_anomaly_timestamp', 'extreme_value_in_window', 'global_p95', 'global_p5']]
else:
    anomaly_summary = pd.DataFrame(columns=['cmdb_id', 'log_name', 'anomaly_count', 'direction', 'earliest_anomaly_timestamp', 'extreme_value_in_window', 'global_p95', 'global_p5'])

# Process error_logs: filter to incident window and prepare samples
error_logs_window = error_logs[(error_logs['timestamp'] >= start) & (error_logs['timestamp'] <= end)].copy()
error_log_count = error_logs_window.shape[0]
if not error_logs_window.empty:
    error_logs_window['timestamp_iso'] = error_logs_window['timestamp'].dt.tz_convert('UTC').dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    error_logs_samples = error_logs_window[['timestamp_iso', 'cmdb_id', 'message']].head(20).rename(columns={'timestamp_iso': 'timestamp'})
else:
    error_logs_samples = pd.DataFrame(columns=['timestamp', 'cmdb_id', 'message'])

# Return compact results: anomaly_summary, error_log_count, error_logs_samples
anomaly_summary, error_log_count, error_logs_samples

```
Out[4]:
```
Summary of log analysis for the incident window (2024-01-17T03:12:50Z — 2024-01-17T03:42:50Z):

- Detected anomalies (compact summary): all are "low" anomalies on log.row_count (values < global P5), each group had 2 breach rows and earliest breach at 2024-01-17T03:15:00Z.
  - frontend — log.row_count: extreme=477 (global_p95=2766.8, global_p5=2266.4)
  - currencyservice — log.row_count: extreme=448 (global_p95=2446.8, global_p5=2009.0)
  - cartservice — log.row_count: extreme=129 (global_p95=805.8, global_p5=647.2)
  - recommendationservice — log.row_count: extreme=92 (global_p95=585.0, global_p5=466.6)
  - adservice — log.row_count: extreme=72 (global_p95=463.8, global_p5=367.2)
  - shippingservice — log.row_count: extreme=88 (global_p95=530.8, global_p5=390.8)
  - checkoutservice — log.row_count: extreme=13 (global_p95=110.4, global_p5=55.8)
  - emailservice — log.row_count: extreme=5 (global_p95=36.8, global_p5=18.6)
  - paymentservice — log.row_count: extreme=10 (global_p95=73.6, global_p5=37.2)

- No log.error_count anomalies were detected (no service had log.error_count > global P95).

- Error log files in the incident window: 0 rows (no error_log entries found in the window).

Interpretation: there is a widespread drop in log.row_count across many services around 03:15:00Z, with no corresponding spike in logged errors. This matches the trace row_count drops previously observed and points to reduced traffic or lost requests rather than increased application errors. If you want, I can correlate these drops with metric signals (cpu/mem/diskio) or network indicators for deeper root-cause analysis.

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id       log_name anomaly_count direction earliest_anomaly_timestamp extreme_value_in_window  global_p95  global_p5
0              adservice  log.row_count             2       low       2024-01-17T03:15:00Z                      72       463.8      367.2
1  recommendationservice  log.row_count             2       low       2024-01-17T03:15:00Z                      92       585.0      466.6
2            cartservice  log.row_count             2       low       2024-01-17T03:15:00Z                     129       805.8      647.2
3               frontend  log.row_count             2       low       2024-01-17T03:15:00Z                     477      2766.8     2266.4
4        currencyservice  log.row_count             2       low       2024-01-17T03:15:00Z                     448      2446.8     2009.0
5        shippingservice  log.row_count             2       low       2024-01-17T03:15:00Z                      88       530.8      390.8
6        checkoutservice  log.row_count             2       low       2024-01-17T03:15:00Z                      13       110.4       55.8
7           emailservice  log.row_count             2       low       2024-01-17T03:15:00Z                       5        36.8       18.6
8         paymentservice  log.row_count             2       low       2024-01-17T03:15:00Z                      10        73.6       37.2, 0, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[5]:

import re
import pandas as pd
import numpy as np

# Reuse existing 'trace' if present; otherwise load
try:
    trace
except NameError:
    trace = pd.read_csv("trace.csv")
    trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)

# Ensure timestamp parsed
if trace['timestamp'].dtype == object or pd.api.types.is_integer_dtype(trace['timestamp'].dtype):
    trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)

# Compute global P5 per (cmdb_id, trace_name) using full series (before filtering)
global_p5_df = (
    trace.groupby(['cmdb_id', 'trace_name'], observed=True)['value']
    .quantile(0.05)
    .reset_index()
    .rename(columns={'value': 'global_p5'})
)

# Incident window (UTC)
start = pd.to_datetime('2024-01-17T03:12:50Z', utc=True)
end   = pd.to_datetime('2024-01-17T03:42:50Z', utc=True)

# Filter trace rows to the incident window
trace_window = trace[(trace['timestamp'] >= start) & (trace['timestamp'] <= end)].copy()

# Candidate services and patterns
services = ['checkoutservice', 'currencyservice', 'emailservice', 'productcatalogservice', 'recommendationservice']
to_regex = r'^trace\.to_(' + '|'.join(re.escape(s) for s in services) + r')\.row_count$'
from_pattern = 'from_'

# Find rows matching trace.to_<service>.row_count and extract target_service
trace_window['to_target'] = trace_window['trace_name'].str.extract(to_regex, flags=re.IGNORECASE, expand=False)
to_rows = trace_window[trace_window['to_target'].notna()].copy()
to_rows['target_service'] = to_rows['to_target'].str.lower()

# Find rows where cmdb_id is one of the services and trace_name contains 'from_' (incoming counts)
from_rows = trace_window[trace_window['cmdb_id'].isin(services) & trace_window['trace_name'].str.contains(from_pattern, case=False, na=False)].copy()
from_rows['target_service'] = from_rows['cmdb_id'].str.lower()

# Combine candidate rows
candidates = pd.concat([to_rows, from_rows], ignore_index=True, sort=False)
if candidates.empty:
    # Empty result
    result_df = pd.DataFrame(columns=['target_service','trace_name','anomaly_count','earliest_anomaly_timestamp','min_value_in_window','global_p5','severity'])
    distinct_service_count = 0
else:
    # Attach global_p5 (computed per original cmdb_id, trace_name)
    candidates = candidates.merge(global_p5_df, on=['cmdb_id', 'trace_name'], how='left')

    # Identify low anomalies: value < global_p5
    candidates['is_anom'] = candidates['value'] < candidates['global_p5']

    # Keep only anomaly rows
    anoms = candidates[candidates['is_anom']].copy()

    if anoms.empty:
        result_df = pd.DataFrame(columns=['target_service','trace_name','anomaly_count','earliest_anomaly_timestamp','min_value_in_window','global_p5','severity'])
        distinct_service_count = 0
    else:
        # Aggregate per target_service and trace_name
        agg = (
            anoms
            .groupby(['target_service', 'trace_name'], observed=True)
            .agg(
                anomaly_count = ('value', 'count'),
                earliest_ts = ('timestamp', 'min'),
                min_value_in_window = ('value', 'min'),
                global_p5 = ('global_p5', 'first')
            )
            .reset_index()
        )
        # Compute severity = global_p5 / min_value_in_window (guard div by zero)
        agg['severity'] = agg.apply(lambda r: (r['global_p5'] / r['min_value_in_window']) if pd.notna(r['min_value_in_window']) and r['min_value_in_window'] != 0 else np.inf, axis=1)

        # Format earliest timestamp
        agg['earliest_anomaly_timestamp'] = pd.to_datetime(agg['earliest_ts']).dt.tz_convert('UTC').dt.strftime('%Y-%m-%dT%H:%M:%SZ')
        agg = agg.drop(columns=['earliest_ts'])

        # Reorder columns per requirement
        result_df = agg[['target_service','trace_name','anomaly_count','earliest_anomaly_timestamp','min_value_in_window','global_p5','severity']]

        # Sort by severity desc then anomaly_count desc and limit to top 10
        result_df = result_df.sort_values(['severity','anomaly_count'], ascending=[False, False]).head(10).reset_index(drop=True)

        # Number of distinct candidate services that show at least one such 'low' anomaly
        distinct_service_count = anoms['target_service'].nunique()

# Output compact results: result_df and distinct_service_count
result_df, distinct_service_count

```
Out[5]:
```
Summary of trace row_count anomalies for the incident window 2024-01-17T03:12:50Z — 2024-01-17T03:42:50Z:

- Number of distinct candidate services with at least one low trace.row_count anomaly: 5 (checkoutservice, currencyservice, emailservice, productcatalogservice, recommendationservice).

- Top anomalous trace metrics (sorted by severity):  
  1. recommendationservice — trace.to_recommendationservice.row_count: anomaly_count=6; earliest=2024-01-17T03:15:00Z; min=4.0; global_p5=466.6; severity=116.65  
  2. checkoutservice — trace.to_checkoutservice.row_count: anomaly_count=6; earliest=2024-01-17T03:15:00Z; min=3.0; global_p5=157.8; severity=52.60  
  3. currencyservice — trace.to_currencyservice.row_count: anomaly_count=6; earliest=2024-01-17T03:15:00Z; min=3.0; global_p5=64.8; severity=21.60  
  4. productcatalogservice — trace.to_productcatalogservice.row_count: anomaly_count=8; earliest=2024-01-17T03:15:00Z; min=4.0; global_p5=46.2; severity=11.55  
  5. checkoutservice — trace.from_frontendservice.row_count: anomaly_count=2; earliest=2024-01-17T03:15:00Z; min=3.0; global_p5=18.6; severity=6.20  
  6. checkoutservice — trace.from_checkoutservice.row_count: anomaly_count=2; earliest=2024-01-17T03:15:00Z; min=26.0; global_p5=157.8; severity≈6.07  
  7. emailservice — trace.from_root.row_count: anomaly_count=2; earliest=2024-01-17T03:15:00Z; min=4.0; global_p5=20.8; severity=5.20  
  8. productcatalogservice — trace.from_root.row_count: anomaly_count=2; earliest=2024-01-17T03:15:00Z; min=4.0; global_p5=20.8; severity=5.20  
  9. recommendationservice — trace.from_root.row_count: anomaly_count=2; earliest=2024-01-17T03:15:00Z; min=4.0; global_p5=20.8; severity=5.20  
 10. productcatalogservice — trace.from_recommendationservice.row_count: anomaly_count=2; earliest=2024-01-17T03:15:00Z; min=92.0; global_p5=466.6; severity≈5.07

Interpretation: there was a pronounced drop in trace row_count around 03:15:00Z affecting incoming and to-<service> counts across multiple core services, with the strongest relative drops seen for requests to recommendationservice (very high severity). This indicates a widespread reduction in trace/throughput toward these services during the window.

The original code execution output of IPython Kernel is also provided below for reference:

(          target_service                                  trace_name  anomaly_count earliest_anomaly_timestamp  min_value_in_window  global_p5    severity
0  recommendationservice    trace.to_recommendationservice.row_count              6       2024-01-17T03:15:00Z                  4.0      466.6  116.650000
1        checkoutservice          trace.to_checkoutservice.row_count              6       2024-01-17T03:15:00Z                  3.0      157.8   52.600000
2        currencyservice          trace.to_currencyservice.row_count              6       2024-01-17T03:15:00Z                  3.0       64.8   21.600000
3  productcatalogservice    trace.to_productcatalogservice.row_count              8       2024-01-17T03:15:00Z                  4.0       46.2   11.550000
4        checkoutservice        trace.from_frontendservice.row_count              2       2024-01-17T03:15:00Z                  3.0       18.6    6.200000
5        checkoutservice        trace.from_checkoutservice.row_count              2       2024-01-17T03:15:00Z                 26.0      157.8    6.069231
6           emailservice                   trace.from_root.row_count              2       2024-01-17T03:15:00Z                  4.0       20.8    5.200000
7  productcatalogservice                   trace.from_root.row_count              2       2024-01-17T03:15:00Z                  4.0       20.8    5.200000
8  recommendationservice                   trace.from_root.row_count              2       2024-01-17T03:15:00Z                  4.0       20.8    5.200000
9  productcatalogservice  trace.from_recommendationservice.row_count              2       2024-01-17T03:15:00Z                 92.0      466.6    5.071739, 5)```
```